In [7]:
import scanpy as sc
import celltypist
import time
import numpy as np
import pandas as pd
import os
from celltypist import models
import cosg
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(8,8), format='pdf')
import matplotlib as mpl
import matplotlib.font_manager as fm
mpl.rcParams['pdf.fonttype'] = 42
fm.fontManager.addfont("fonts/arial.ttf")
mpl.rcParams['font.family'] = 'arial'
mpl.rcParams['lines.linewidth'] = 0.5

In [2]:
adata = sc.read_h5ad('/home/liyanguo/MyImmuCell/05_Ref_Atlas_subpopulation/Level2_Refine_R1/Celltype_L1_L2_Refine_R1.h5ad')

In [3]:
adata

AnnData object with n_obs × n_vars = 717723 × 38606
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'SampleID', 'DonorID', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop', 'percent_dna_repair', 'percent_ieg', 'percent_hemo', 'S.Score', 'G2M.Score', 'Phase', 'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'predicted.celltype.l1', 'predicted.celltype.l2', 'predicted.celltype.l3', 'scDblFinder.class', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'High_quality', 'Batch', 'L1_leiden_scVI_3', 'L1_leiden_scVI_4', 'L1_leiden_scVI_5', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'VJ_1_locus', 'VJ_1_v_call', 'VJ_1_j_call', 'VJ_1_junction_aa', 'VJ_1_junction', 'VJ_1_umi_count', 'VDJ_1_locus', 'VDJ_1_v_call', 'V

In [4]:
adata.obs['Celltype_L1_L2_Refine'].value_counts()

Celltype_L1_L2_Refine
NDNs                         240522
CD4+ T cells                 158168
Non-MAIT/NKT CD8+ T cells    114555
NK cells                      79020
Classical monocytes           44552
B cells                       28220
MAIT                          17983
Non-classical monocytes       15652
γδ T cells                     5109
cDCs                           3096
Proliferative T/NK             2341
Plamsa cells                   1842
Basophils                      1769
pDCs                           1699
LDNs                           1645
Platelets                      1189
HSPC                            261
Mast                             58
iNKT                             42
Name: count, dtype: int64

# Avoid the doublet cell potentially

In [39]:
adata=adata[adata.obs['scDblFinder.class']=='singlet',:].copy()

In [40]:
sc.pp.normalize_total(adata,target_sum = 1e4)
sc.pp.log1p(adata)

# Use `celltypist.train` to quickly train a rough CellTypist model.

In [41]:
%%time
model= celltypist.train(adata,
                         labels='Celltype_L1_L2_Refine', n_jobs = 48,
                         max_iter = 10,
                         use_SGD = True)

🍳 Preparing data before training
✂️ 4242 non-expressed genes are filtered out
🔬 Input data has 665471 cells and 34364 genes
⚖️ Scaling input data
🏋️ Training data using SGD logistic regression
⚠️ Warning: it may take a long time to train this dataset with 665471 cells and 34364 genes, try to downsample cells and/or restrict genes to a subset (e.g., hvgs)
✅ Model training done!


CPU times: user 1h 18min 55s, sys: 2min 28s, total: 1h 21min 23s
Wall time: 18min 39s


In [43]:
model.write('/home/liyanguo/tmp.pkl')

## Feature selection for better model training.

In [44]:
model = models.Model.load(model = '/home/liyanguo/tmp.pkl')

In [47]:
top_n = 200

gene_index = np.argpartition(
    np.abs(model.classifier.coef_),
    -top_n,
    axis = 1
)[:, -top_n:]
gene = np.unique(model.features[gene_index])
print(f"Number of genes selected: {len(gene)}")

Number of genes selected: 1947


## COSG Feature selection for better model training.

In [48]:
#cosg差异
df_tmp = cosg.cosg(adata, groupby='Celltype_L1_L2_Refine',key_added='cosg',
          mu=10,n_genes_user=100,remove_lowly_expressed=True,
         )

In [49]:
df_tmp = pd.DataFrame(adata.uns['cosg']['names'])

In [50]:
cosg_gene = df_tmp.values.flatten()

In [51]:
Feature_selected = np.unique(np.hstack((gene, cosg_gene)))

In [52]:
print(f"Number of genes selected: {len(Feature_selected)}")

Number of genes selected: 2607


## Remove Noise gene: Erythroid

In [53]:
adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))

In [54]:
erythrocyte_gene = adata.var_names[ adata.var["hb"]]

In [55]:
erythrocyte_gene.values

array(['HBEGF', 'HBS1L', 'HBB', 'HBD', 'HBG1', 'HBG2', 'HBE1', 'HBZ',
       'HBM', 'HBA2', 'HBA1', 'HBQ1'], dtype=object)

In [56]:
Feature_selected = np.setdiff1d(Feature_selected, erythrocyte_gene)

In [57]:
print(f"Number of genes selected: {len(Feature_selected)}")

Number of genes selected: 2606


## Remove Noise gene: BCR\TCR

In [58]:
tcrbcr_gene = pd.read_table('tcr_bcr_gene.txt',header=None)

In [59]:
tcrbcr_gene = tcrbcr_gene.values.flatten()

In [60]:
Feature_selected = np.setdiff1d(Feature_selected, tcrbcr_gene)

In [61]:
print(f"Number of genes selected: {len(Feature_selected)}")

Number of genes selected: 2539


## Add My marker

In [62]:
marker_dict1={
    'Immune cell': ['PTPRC'],

    'Lineage':['CD7','CD3E',
               'IL7R',
               'SPON2','KLRF1',
               'CD79A','MS4A1',
               'CD14','FCGR3A',
               'CSF3R','FCGR3B'
              ],
    
    'HSPC':['CD34','SPINK2','CYTL1','PROM1','SMIM24','EGFL7',
            'SOX4','KIT', 'DNTT','ETV6','MCL1','STAT5A','CD48'],
    'Basophil':['HDC','GATA2', 'FCER1A', 'IL3RA','ENPP3','MS4A2','IL4'],
    'Eosinophil':['ALOX15','SIGLEC10','SIGLEC8','LMO4','EPX','ITGA1','CCR3',],
    'Mast cell':['FCER1A','MS4A2','KIT'],

    'LDNs':['FCGR3B','CSF3R','MME','G0S2','MNDA',],
    'NDNs':['CEACAM8','LTF','BPI','MPO','ELANE'],
    
    'Classical monocytes' :['CD14','LYZ','VCAN','FCN1',],
    'Non-classical monocytes' :['FCGR3A','CDKN1C','TCF7L2','CSF1R',],
    'DC':['ENHO','CD1C','HLA-DQA1','FCER1A','CLEC10A','CLEC9A','FLT3'],
    'pDC':['CLEC4C','IL3RA'],
    
    'T naive': ['CD3D','CD3G','LEF1','IL7R','CCR7', 'TCF7','SELL','BACH2'],
    'CD4+ T': ['CD4','MAL','RCAN3','IL6ST','TRAT1','CAMK4'],
    'TRAV1-2- CD8+ T': ['CD8A', 'CD8B','LINC02446','CCL5','GZMH','GZMK','ZNF683','THEMIS'],
    'γδ T':['TRDV2','TRGV9','TRDC','KLRG1','TRGC1','TRGC2','CCL5','CST7',],
    'MAIT':['SLC4A10','KLRB1','TRAV1-2','RORA','CXCR6'],
    'Treg':['FOXP3','CTLA4','IL2RA','TIGIT','RTKN2','STAM'],

    'NK': ['KLRD1','GNLY','PRF1','GZMB','CD244','CD247',
           'IL2RB','XCL1','XCL2'],
    'CD56dim':['FCGR3A','SPON2',],#CD16+
    'CD56bright':['IL7R','GZMK','NCAM1','GATA3'],#CD16-/low
    'Adaptive NK':['KLRC2','B3GAT1',],

    'Lymphocyte relate':['RUNX1','RUNX2','KLRF1','KLRC1',#KLRC1=NKG2AA抑制   NKp80=KLRF1
                         'KLRD1','KLRG1','EOMES',
                        'NCR2','NCR3','SYNE1'],#KLRD1=CD94抑制 KLRG1抑制

    'ILC': ['IL7R','LTB','RGS1','TNFSF10','PLCG2','RUNX1','TOX','FLT3'],#lin-
    'pILC':['NFIL3',],
    'ILC1':['IL2RB','TBX21','NCR1',],#CD56-
    'ILC2':['GATA3','IL2RA','PTGDR2','KLRG1','ZBTB16','RORA','MAF'],#lin- CD4-
    'ILC3-NCR+':['AHR','KIT','RORC','NCR1','NCR2','RUNX2',],#mature ILC #lin- CD56+ CD4-
    'LTi':['CCR7','KIT','ITGA4',],#mature ILC #lin- CD56-
    'ILCreg':['SOX4',],#KLRG1- 
    
    'NKT':['TRAV24','TRBV28','CLDND1'], 
    'DN T':['FXYD2','NUCB2','MYB',],
    'T Develop':['SPI1','ZBTB7B','RUNX3'],
    'T activation': ['CD69', 'CD38'],

    'B naive': ['CD79A','TCL1A',],
    'Transitional B': ['EBF1','BACH2','PAX5', 'MSI2',],
    'Atypical memory B': ['TBX21','ITGAX','FCRL5','SIGLEC6','SOX5'],
    'Plasma':['JCHAIN','IGHA1','TNFRSF17','MZB1','CD38','XBP1', 'PRDM1'],
    'CD5+ B':['IGHV3-7','LEF1','CTLA4','CD5'],

    'Platelet':['PF4','PPBP','GP9','CAVIN2'],
    'RBC':['HBB','HBA2','BLVRB'],
    'Proliferative signal':['MKI67','TOP2A','STMN1'],
    'Other':['ITGAM','ANK3','PRKCA','BCL6','TGFB1','CXCR4','GATA1','ATXN1','SELPLG',
             'ITGA6','CSF1','CSF1R','CSF2','ETS1','CD44','IRF4','STAT3','BATF','TGFBR1','TGFBR2','CD24']
}

In [5]:
marker_dict2={
    'B': ['MS4A1','CD79A','BANK1','CD22','FCRL1',],
    'Plasma|Plasmablast':['TNFRSF17','JCHAIN','IGHA2','MZB1'],
    
    'CD4+ T':['CD4','TCF7','MAL','IL7R','RCAN3','DGKA',],
    'CD8+ T':['CD8A','CD8B','LINC02446','CCL5','THEMIS'],
    'MAIT':['SLC4A10','TRAV1-2','KLRB1',],
    'Platelet':['TUBB1','CAVIN2','PF4',],
    'NKT':['TRAV24','TRBV28','CLDND1','DZIP3'],
    'NK': ['KLRF1', 'SPON2','GNLY', 'GZMB','PRF1'],
    'non-NK ILC': ['FSTL4', 'HPGDS','PKIB', 'PTGDR2','SCART1'],
    'γδ T':['TRDV2','TRGV9','TRGC1','TRGC2','CCL5','KLRC1'],
    
    'HSC':['CD34','NPR3','SMIM24','SPINK2', 'CRHBP'],
    'Dendritic':['ENHO','CLEC10A','CD1C','FCER1A','FLT3'],
    'Classical monocyte':['FCN1','VCAN','LRP1','CD14','MS4A6A',],
    'Non-classical monocyte':['CDKN1C','CSF1R','CKB','NEURL1','LYPD2'],
    'LDNs':['CEACAM8','LCN2','BPI','LTF','CRISP3'],
    'NDNs':['CSF3R','NAMPT','FCGR3B','IFITM2','SLC25A37'],
    'Eosinophil':['EPX', 'SIGLEC8', 'IL5RA'],
    'Basophil':['HDC','GATA2','MS4A2','AKAP12','CLC'],
    'pDC':['SERPINF1','PTPRS','LILRA4','TNFRSF21','PLD4',],
    
    'Proliferative signal':['MKI67','TOP2A','STMN1'],
}

In [64]:
all_items = []
for values in marker_dict1.values():
    all_items.extend(values)
dict1 = np.unique(np.array(all_items))

In [65]:
all_items = []
for values in marker_dict2.values():
    all_items.extend(values)
dict2 = np.unique(np.array(all_items))

In [66]:
Feature_selected = np.unique(
    np.hstack((Feature_selected, dict2,dict1))
)

In [67]:
print(f"Number of genes selected: {len(Feature_selected)}")

Number of genes selected: 2584


## Output

In [68]:
Feature_selected_df = pd.DataFrame({'gene': Feature_selected})

In [69]:
Feature_selected_df.to_csv("celltypist_Feature_selected.csv")

# Full Model training

In [70]:
Feature_selected = pd.read_csv("celltypist_Feature_selected.csv",index_col=0)

In [71]:
Feature_selected

,gene
0,AARSD1
1,AATBC
2,ABCA13
3,ABCA2
4,ABCB1
...,...
2579,ZNF831
2580,ZNF836
2581,ZNRF1
2582,ZWINT


In [72]:
Feature_selected = Feature_selected['gene'].values

In [73]:
adata = sc.read_h5ad('/home/liyanguo/MyImmuCell/05_Ref_Atlas_subpopulation/Level2_Refine_R1/Celltype_L1_L2_Refine_R1.h5ad')

## Avoid the doublet cell potentially

In [74]:
adata=adata[adata.obs['scDblFinder.class']=='singlet',:].copy()

In [75]:
sc.pp.normalize_total(adata,target_sum = 1e4)
sc.pp.log1p(adata)

In [76]:
print(f"Number of genes selected: {len(Feature_selected)}")

Number of genes selected: 2584


In [77]:
adata = adata[:, adata.var_names.isin(Feature_selected)].copy()

In [78]:
adata.shape

(665471, 2584)

In [79]:
%%time
model= celltypist.train(adata,
                         labels='Celltype_L1_L2_Refine', n_jobs = 48,
                         check_expression = False,
                         max_iter = 100)

🍳 Preparing data before training
🔬 Input data has 665471 cells and 2584 genes
⚖️ Scaling input data
🏋️ Training data using logistic regression
✅ Model training done!


CPU times: user 5h 43min 18s, sys: 9.8 s, total: 5h 43min 27s
Wall time: 19min 43s


In [80]:
# Save the model.
model.write('/home/liyanguo/MyImmuCell/05_Ref_Atlas_subpopulation/Level2_Refine_R1/model_Ref_Atlas_L1_L2_HVG.pkl')

In [81]:
os.system('cp /home/liyanguo/MyImmuCell/05_Ref_Atlas_subpopulation/Level2_Refine_R1/model_Ref_Atlas_L1_L2_HVG.pkl ~/.celltypist/data/models/')

0

## Cell type-driving genes

In [82]:
model = models.Model.load(model = '/home/liyanguo/MyImmuCell/05_Ref_Atlas_subpopulation/Level2_Refine_R1/model_Ref_Atlas_L1_L2_HVG.pkl')

In [84]:
marker_dict={}
for i in model.cell_types:
    top_n=5
    top_n_genes = model.extract_top_markers(i, top_n)
    marker_dict[i]=top_n_genes

In [85]:
sc.pl.dotplot(
    adata,
    groupby='Celltype_L1_L2_Refine',
    save="celltypist_driving_Feature",
    var_names=marker_dict,
    standard_scale="var",
    swap_axes=False,
    #dot_min=0.1,
    show=False,
    cmap='Spectral_r'
)

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

{'mainplot_ax': <Axes: >,
 'gene_group_ax': <Axes: >,
 'size_legend_ax': <Axes: title={'center': 'Fraction of cells\nin group (%)'}>,
 'color_legend_ax': <Axes: title={'center': 'Mean expression\nin group'}>}

# Vis

In [24]:
adata.obs['Celltype_L1_L2_Refine']=adata.obs['Celltype_L1_L2_Refine'].cat.reorder_categories([
'HSPC', 
'LDNs',
'NDNs',
'Basophils',
'Mast',
'CD4+ T cells',
'Non-MAIT/NKT CD8+ T cells',
'MAIT',
'γδ T cells',
'iNKT',
'NK cells',
'Proliferative T/NK',
'B cells',
'Plamsa cells',
'Classical monocytes',
'Non-classical monocytes',
'cDCs',
'pDCs',
'Platelets'
])

In [27]:
marker_dict2=['CD34','SPINK2',
              'CEACAM8','CEBPE',
              'CSF3R','CEBPB','FCGR3B',#NDNs
              'IL4','ENPP3','FCER1A','IL3RA','KIT',
              'TPSD1',
              'CD3D','CD4','CD8A','CD8B',
              'SLC4A10','TRAV1-2','KLRB1',
              'TRDV2','TRGV9','TRDV1',
              'TRAV24',
              'KLRF1','GNLY',
              'MKI67',
              'MS4A1','CD79A',
              'TNFRSF17','JCHAIN',
              'CD14','VCAN',
              'CDKN1C','CSF1R',
              'CLEC10A','CD1C',
              'CLEC4C','LILRA4',
              'PF4','PPBP'
]

In [28]:
sc.pl.dotplot(
    adata,
    groupby='Celltype_L1_L2_Refine',
    save="Reference_key_marker",
    var_names=marker_dict2,
    standard_scale="var",
    swap_axes=False,
    #dot_min=0.1,
    show=False,
    cmap='Spectral_r'
)

maxp pruned
LTSH dropped
cmap pruned
kern pruned
post pruned
PCLT dropped
JSTF dropped
DSIG dropped
GSUB pruned
glyf pruned
Added gid0 to subset
Added first four glyphs to subset
Closing glyph list over 'GSUB': 62 glyphs before
Glyph names: ['.notdef', '.null', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'P', 'R', 'S', 'T', 'V', 'Y', 'a', 'c', 'delta', 'e', 'eight', 'f', 'five', 'four', 'g', 'gamma', 'h', 'hyphen', 'i', 'l', 'm', 'n', 'nine', 'nonmarkingreturn', 'o', 'one', 'p', 'parenleft', 'parenright', 'percent', 'period', 'plus', 'r', 's', 'seven', 'six', 'slash', 'space', 't', 'three', 'two', 'u', 'v', 'x', 'y', 'zero']
Glyph IDs:   [0, 1, 2, 3, 8, 11, 12, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 53, 54, 55, 57, 60, 68, 70, 72, 73, 74, 75, 76, 79, 80, 81, 82, 83, 85, 86, 87, 88, 89, 91, 92, 303, 534]
Closed glyph list over 'GSUB': 62 glyphs after
Glyph names: ['.notdef', '.null', '

{'mainplot_ax': <Axes: >,
 'size_legend_ax': <Axes: title={'center': 'Fraction of cells\nin group (%)'}>,
 'color_legend_ax': <Axes: title={'center': 'Mean expression\nin group'}>}

In [86]:
import session_info
session_info.show()